# Generate QA Dataset for an Existing Corpus

Creates (or recreates) a `train_questions.parquet` for an **already-indexed**
corpus so it can be evaluated with `rag_evaluation.ipynb`.

**Workflow:**
1. Set `NAME` to the target collection (must already have `wiki_corpus.parquet`)
2. Choose QA sources & balancing options
3. Run all cells — loads, enriches, balances, saves
4. Open `rag_evaluation.ipynb` with the same `NAME` and evaluate

In [1]:
from pathlib import Path
import pandas as pd
from config import DATA_DIR, CACHE_DIR

# ── Target collection (must already have wiki_corpus.parquet) ────────────────
NAME = "wiki_full"
COLLECTION_ROOT = Path(DATA_DIR) / NAME
WIKI_PARQUET   = COLLECTION_ROOT / "wiki_corpus.parquet"
QUESTIONS_PATH = COLLECTION_ROOT / "popqa.parquet"
    
# ── QA sources (HuggingFace config names) ────────────────────────────────────
QA_DATASETS = ["popqa"]  # Datasets to pull questions from (must be in QUESTIONS_PATH)
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"

# ── Balancing ────────────────────────────────────────────────────────────────
BALANCE = True                               # Whether to balance questions across popularity deciles
TARGET_PER_DECILE = 300              # None → downsample to smallest decile

# ── Synthetic generation (optional) ─────────────────────────────────────────
GENERATE_SYNTHETIC = False
QUESTIONS_PER_DECILE = 300
MODEL_NAME = "gpt-4.1-nano"

# ── Sanity check ─────────────────────────────────────────────────────────────
assert WIKI_PARQUET.exists(), f"Corpus not found: {WIKI_PARQUET}"
print(f"✓ Collection: {NAME}")
print(f"  Corpus:     {WIKI_PARQUET}  ({WIKI_PARQUET.stat().st_size / 1e9:.2f} GB)")
print(f"  QA sources: {QA_DATASETS}")
print(f"  Balance:    {BALANCE}  |  Synthetic: {GENERATE_SYNTHETIC}")

✓ Collection: wiki_full
  Corpus:     /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/wiki_corpus.parquet  (9.88 GB)
  QA sources: ['popqa']
  Balance:    True  |  Synthetic: False


In [2]:
from scripts.prepare_qa_dataset import prepare_qa_dataset

qa_df = prepare_qa_dataset(
    qa_datasets=QA_DATASETS,
    popularity_dataset=POPULARITY_DATASET,
    output_path=QUESTIONS_PATH,
    balance=BALANCE,
    target_per_decile=TARGET_PER_DECILE,
    generate_synthetic_flag=GENERATE_SYNTHETIC,
    corpus_path=WIKI_PARQUET,  # Always filter to corpus
    questions_per_decile=QUESTIONS_PER_DECILE,
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
)


📥 Loading QA datasets: ['popqa']


/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


  popqa: 13,811 questions
✓ Merged QA: 13,811 questions from 1 dataset(s)

📊 Combined QA: 13,811 questions

🔍 Filtering QA to match corpus...
  ✓ All 13,811 questions exist in corpus

📈 Enriching with popularity deciles...
Loading popularity data for decile calculation...
  Calculating global deciles...
✓ Enriched: 13,811 questions with deciles

⚖️ Balancing...
  Balancing to 300 questions per decile...
✓ Balanced: 2,805 questions (300 × 10 deciles)
  Distribution:
decile
0    133
1    272
2    300
3    300
4    300
5    300
6    300
7    300
8    300
9    300
Name: count, dtype: int64

✅ Done: 2,805 questions saved to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/popqa.parquet
Distribution:
decile
0    133
1    272
2    300
3    300
4    300
5    300
6    300
7    300
8    300
9    300
Name: count, dtype: int64
Sources:
dataset
popqa    2805
Name: count, dtype: int64


In [3]:
# ── Quick inspection ──────────────────────────────────────────────────────────
print(f"Saved: {QUESTIONS_PATH}")
print(f"Total: {len(qa_df):,} questions\n")

if "decile" in qa_df.columns:
    print("Per-decile distribution:")
    print(qa_df["decile"].value_counts().sort_index())

if "dataset" in qa_df.columns:
    print(f"\nSources:")
    print(qa_df["dataset"].value_counts())

display(qa_df.sample(5, random_state=42))

Saved: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/popqa.parquet
Total: 2,805 questions

Per-decile distribution:
decile
0    133
1    272
2    300
3    300
4    300
5    300
6    300
7    300
8    300
9    300
Name: count, dtype: int64

Sources:
dataset
popqa    2805
Name: count, dtype: int64


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,is_synthetic,decile
1091,5923656,Who is the author of The Empire?,[D. C. Moore],30996440,The Empire (play),48.125000,3.294085e+06,popqa,False,4
1041,5626437,Who was the producer of Romance on the Range?,[Joseph Kane],23691047,Romance on the Range (film),50.125000,3.237574e+06,popqa,False,4
1521,4564108,In what country is Institute of Technological ...,[Sri Lanka],13849571,Institute of Technological Studies,71.812500,2.821455e+06,popqa,False,5
2504,5958980,Who was the screenwriter for The Tape?,[Larry David],5072856,The Tape,1323.979167,6.343111e+05,popqa,False,8
1518,345035,In what city was Vladimir Korotkov born?,[Moscow],34768366,Vladimir Korotkov (tennis),83.895833,2.768027e+06,popqa,False,5


In [4]:
# ── Verify overlap with corpus ────────────────────────────────────────────────
corpus_ids = set(pd.read_parquet(WIKI_PARQUET, columns=["wikipedia_id"])["wikipedia_id"].astype(int))
qa_ids     = set(qa_df["wikipedia_id"].astype(int))

in_corpus = qa_ids & corpus_ids
missing   = qa_ids - corpus_ids

print(f"QA doc IDs in corpus: {len(in_corpus):,} / {len(qa_ids):,}  ({100 * len(in_corpus) / len(qa_ids):.1f}%)")
if missing:
    print(f"⚠️  {len(missing):,} QA doc IDs NOT in corpus — these questions can never be answered correctly")
else:
    print("✓ All QA documents exist in the corpus")

QA doc IDs in corpus: 2,752 / 2,752  (100.0%)
✓ All QA documents exist in the corpus
